# Pipeline walkthrough: document → sections → generator → verifier → judge

This notebook runs the real project code step by step on one document and visualises what each
step does:

| Part | Step | Code | Model |
|---|---|---|---|
| 1 | Read & clean the document | `quizgen.extract` | - |
| 2 | Split it into sections (chunking) | `quizgen.extract.split_sections` | - |
| 3 | Generate questions | `quizgen.generate.generate_for_section` | generator (model 1) |
| 4 | Verify them (quote check + answer-back) | `quizgen.verify.verify_one` | verifier (model 2) |
| 5 | Grade them with the rubric (LLM-as-judge) | `eval.judge.judge_one` | judge (model 1) |
| 6 | Cost and time per step | `Usage` of each model | - |
| 7 | Explore a finished evaluation run (`reports/eval`) | files written by `eval.run_eval` | - |

**Cost.** With `USE_REAL_MODELS = False` every model is `FakeLLM`: free and offline, but its questions
are meaningless (fill-in-the-blank from a sentence). Set it to `True` to call the models numbered in
`.env` (`LLM_MODEL_1`, `LLM_MODEL_2`); with the defaults below (2 sections x 3 questions) a run costs
well under $0.01. Part 7 makes no model call at all.

**Copyright.** Outputs show text of the document. Before committing, run
*Kernel → Restart & Clear All Outputs* when `DOC` is a course document (`tests/test_notebook.py`
fails if the committed notebook contains outputs).

In [ ]:
import json
import os
import sys
import time
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)                      # .env and relative paths are read from the repo root
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import FancyBboxPatch

from eval.judge import judge_one
from quizgen.dedupe import mark_duplicates
from quizgen.extract import (MAX_TOKENS_PER_SECTION, MIN_TOKENS_PER_SECTION, back_matter_kind,
                             front_matter_kind, is_slide_deck, load_pages, split_sections,
                             strip_boilerplate)
from quizgen.generate import generate_for_section
from quizgen.llm import FakeLLM, extract_json, model_for_role
from quizgen.schema import RUBRIC_KEYS
from quizgen.verify import find_context, quote_in_source, verify_one

pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("repo root:", ROOT)

### Settings

In [ ]:
DOC = "data/samples/transport_layer_tcp_udp.md"   # e.g. "data/private/Chapter4_PaaS.pdf"
USE_REAL_MODELS = False     # False: FakeLLM (free). True: models from .env (paid / local Ollama)
GENERATOR, VERIFIER, JUDGE = 1, 2, 1                 # model numbers in .env (LLM_MODEL_<n>)
STRATEGY = "v1"             # v1 zero-shot | v2 few-shot | v3 plan -> generate
N_PER_SECTION = 3
MAX_SECTIONS = 2            # sections sent to the models in this demo
EXCLUDE = "Acknowledgement|Policies for students|COURSE HANDOUT|EXAM DOMAINS"   # like --exclude-sections ("" = none)
RUN_DIR = "reports/eval"    # finished run explored in part 7
FIG_DIR = Path("reports/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)

class Recorder:
    # wraps a model and keeps every prompt sent and every raw answer received, to show them below
    def __init__(self, llm):
        self.llm, self.calls = llm, []
    name = property(lambda self: self.llm.name)
    usage = property(lambda self: self.llm.usage)
    def cost_usd(self):
        return self.llm.cost_usd()
    def chat(self, messages, *args, **kwargs):
        out = self.llm.chat(messages, *args, **kwargs)
        self.calls.append({"messages": messages, "response": out})
        return out

def model(role, n):
    return Recorder(model_for_role(role, n) if USE_REAL_MODELS else FakeLLM())

def snapshot(llm):
    u = llm.usage
    return (u.calls, u.prompt_tokens, u.completion_tokens, llm.cost_usd())

LEDGER = {}   # consumption of every step, summarised in part 6
def show_usage(step, llm, before, seconds, n_items=0, item="question"):
    # consumption of one step: model calls, tokens, cost and time (also per item)
    d = [a - b for a, b in zip(snapshot(llm), before, strict=True)] if llm else [0, 0, 0, 0.0]
    row = {"step": step, "model": llm.name if llm else "-", "calls": d[0], "tokens in": d[1], "tokens out": d[2],
           "cost $": round(d[3], 5), "seconds": round(seconds, 2), "items": n_items}
    if n_items:
        row |= {"tokens / item": round((d[1] + d[2]) / n_items), "$ / 100 items": round(100 * d[3] / n_items, 4),
                "s / item": round(seconds / n_items, 2)}
    LEDGER[step] = row
    print(f"CONSUMPTION of '{step}' ({n_items} {item}s):")
    display(pd.DataFrame([row]).set_index("step"))

def show_calls(llm, start, count=1, max_chars=1200):
    # output of a step: the exact prompt sent to the model and its raw answer
    for c in llm.calls[start:start + count]:
        for m in c["messages"]:
            t = m["content"]
            print(f"----- {m['role']} prompt ({len(t):,} characters){' - first part only' if len(t) > max_chars else ''}")
            print(t[:max_chars], "\n")
        try:
            raw = json.dumps(json.loads(extract_json(c["response"])), ensure_ascii=False, indent=1)
        except ValueError:
            raw = c["response"]
        print(f"----- raw answer of {llm.name}\n{raw[:2 * max_chars]}\n")
    if not llm.calls[start:start + count]:
        print("(no model call in this step)")

gen_llm, ver_llm, judge_llm = model("generator", GENERATOR), model("verifier", VERIFIER), model("judge", JUDGE)
print("generator:", gen_llm.name, "| verifier:", ver_llm.name, "| judge:", judge_llm.name)

## 0. The pipeline at a glance

In [ ]:
steps = [("Document\nPDF / PPTX / MD", "#e8eef7"), ("Extract & clean\nboilerplate, cover,\nsummary slides", "#e8eef7"),
         ("Chunk\n300-1,500 tokens", "#e8eef7"), ("Generate\nmodel 1 → JSON", "#fdebd3"),
         ("Verify\nquote check (code)\n+ answer-back\n(model 2)", "#fdebd3"), ("Dedupe\n& export", "#e8eef7"),
         ("Judge (eval only)\nmodel 1, 5-criterion\nrubric", "#e3f1e3"), ("Human rating\n→ kappa vs judge", "#e3f1e3")]
fig, ax = plt.subplots(figsize=(14, 2.6)); ax.axis("off")
w, gap = 1.55, 0.25
for i, (label, color) in enumerate(steps):
    x = i * (w + gap)
    ax.add_patch(FancyBboxPatch((x, 0), w, 1, boxstyle="round,pad=0.04", fc=color, ec="#555"))
    ax.text(x + w / 2, 0.5, label, ha="center", va="center", fontsize=8.5)
    if i < len(steps) - 1:
        ax.annotate("", xy=(x + w + gap, 0.5), xytext=(x + w, 0.5), arrowprops=dict(arrowstyle="->", color="#555"))
ax.set_xlim(-0.1, len(steps) * (w + gap)); ax.set_ylim(-0.15, 1.15)
ax.set_title("blue = code only · orange = production models · green = evaluation only", fontsize=9)
fig.savefig(FIG_DIR / "pipeline.png", bbox_inches="tight"); plt.show()

## 1. Read and clean the document

* **PDF**: text rebuilt from the position of every character (a space is inserted where the PDF has
  only a visual gap, e.g. Vietnamese decks); **PPTX**: one page per slide (pictures and code
  screenshots have no text); **TXT/MD**: pages split on form feeds.
* Lines repeated on ≥30 % of the pages (headers, footers, course name) and page numbers are removed.
* In slide decks, cover / agenda / objectives / divider slides and summary / exercises / references /
  closing slides are dropped: questions about them are about the lecture, not the subject.

In [ ]:
t0 = time.perf_counter()
raw = load_pages(DOC)
clean = strip_boilerplate(raw)
deck = is_slide_deck(clean)
def kind(p):
    k = (front_matter_kind(p) if deck else None) or back_matter_kind(p)
    return k or ("empty" if not p.text.strip() else "content")

removed = Counter()
for a, b in zip(raw, clean, strict=True):
    kept = set(b.text.split("\n"))
    removed.update(ln.strip() for ln in a.text.split("\n") if ln.strip() and ln not in kept)

pages = pd.DataFrame({"page": [p.number for p in clean], "words": [len(p.text.split()) for p in clean],
                      "kind": [kind(p) for p in clean]})
show_usage("1 extract", None, (), time.perf_counter() - t0, len(raw), "page")
print(f"{len(raw)} pages · detected as {'SLIDE DECK' if deck else 'DOCUMENT'} · "
      f"{sum(removed.values())} boilerplate lines removed")
first = next((p for p in clean if kind(p) == "content"), clean[0])
print(f"\nOUTPUT - page {first.number} after cleaning:\n{first.text[:600]}")
display(pd.DataFrame(removed.most_common(8), columns=["removed line", "times"]))
display(pages["kind"].value_counts().rename("pages").to_frame())

colors = {"content": "#4c72b0", "cover": "#dd8452", "agenda": "#dd8452", "objectives": "#dd8452",
          "divider": "#dd8452", "summary": "#c44e52", "closing": "#c44e52", "empty": "#bbbbbb"}
fig, ax = plt.subplots(figsize=(12, 2.8))
ax.bar(pages["page"], pages["words"], color=[colors[k] for k in pages["kind"]])
from matplotlib.patches import Patch
ax.set(xlabel="page / slide", ylabel="words", title="Words per page after cleaning (orange / red = dropped)")
ax.legend(handles=[Patch(color=colors[k], label=k) for k in pages["kind"].unique()], ncol=4, fontsize=8); fig.savefig(FIG_DIR / "pages.png", bbox_inches="tight"); plt.show()

## 2. Chunking: pages → sections

Slides are short (50-150 tokens), so consecutive slides are **merged** until a section has at least
300 tokens; anything longer than 1,500 tokens is **split**; sections still under 40 tokens are dropped.
Documents are split on their headings first. Tokens are estimated as characters / 3.5.

In [ ]:
import re
t0 = time.perf_counter()
sections = split_sections(raw)
if EXCLUDE:
    skipped = [x.title for x in sections if re.search(EXCLUDE, x.title, re.IGNORECASE)]
    sections = [x for x in sections if not re.search(EXCLUDE, x.title, re.IGNORECASE)]
    for t in skipped:
        print("[exclude]", t[:90])
show_usage("2 chunk", None, (), time.perf_counter() - t0, len(sections), "section")
if sections:
    print(f"OUTPUT - section 0 ({sections[0].n_tokens} tokens, pages {sections[0].page_start}-{sections[0].page_end}):"
          f"\n{sections[0].text[:700]}\n")
sec = pd.DataFrame({"section": range(len(sections)), "pages": [f"{s.page_start}-{s.page_end}" for s in sections],
                    "tokens": [s.n_tokens for s in sections], "title": [s.title[:80] for s in sections]})
display(sec)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, max(2.5, 0.32 * len(sections))), gridspec_kw={"width_ratios": [1, 1.3]})
a1.barh(sec["section"], sec["tokens"], color="#4c72b0"); a1.invert_yaxis()
a1.axvline(MIN_TOKENS_PER_SECTION, ls="--", c="grey"); a1.axvline(MAX_TOKENS_PER_SECTION, ls="--", c="grey")
a1.set(xlabel="estimated tokens", ylabel="section", title="Size of each section (dashed: 300 / 1,500)")
for s_i, s in enumerate(sections):                      # which pages went into which section
    a2.barh(s_i, s.page_end - s.page_start + 1, left=s.page_start - 0.5, color="#4c72b0")
dropped = pages[pages["kind"] != "content"]["page"]
a2.scatter(dropped, [-0.8] * len(dropped), marker="x", c="#c44e52", label="dropped or empty page")
a2.invert_yaxis(); a2.set(xlabel="page / slide", title="Pages covered by each section"); a2.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR / "sections.png", bbox_inches="tight"); plt.show()

## 3. Generate questions (generator)

Each section is sent with the system rules (only use the document, one correct option, a verbatim
`source_quote`, no NOT/EXCEPT questions, an explanation covering every option). The answer is
validated against a JSON schema (retried with the error message if invalid) and the options are
**shuffled**, because models tend to put the correct answer first.

In [ ]:
before, start, t0 = snapshot(gen_llm), len(gen_llm.calls), time.perf_counter()
qs = []
for i, s in enumerate(sections[:MAX_SECTIONS]):
    new = generate_for_section(gen_llm, s, n=N_PER_SECTION, strategy=STRATEGY, seed=i)
    for k, q in enumerate(new):
        q.qid = f"{i}-{k}"
    qs += new
show_usage("3 generate", gen_llm, before, time.perf_counter() - t0, len(qs))
print("OUTPUT - prompt and raw answer for the first section" + (" (v3: plan call, then generate call)" if STRATEGY == "v3" else ""))
show_calls(gen_llm, start, 2 if STRATEGY == "v3" else 1)
by_title = {s.title: s for s in sections}

gq = pd.DataFrame([{"qid": q.qid, "level": q.mcq.level, "question": q.mcq.question,
                    "answer": "ABCD"[q.mcq.answer_index], "source_quote": q.mcq.source_quote[:80]} for q in qs])
display(gq)
if qs:
    ex = qs[0].mcq
    print("Example:", ex.question)
    for j, o in enumerate(ex.options):
        print(f"  {'ABCD'[j]}. {o}{'   <- correct' if j == ex.answer_index else ''}")
    print("Explanation:", ex.explanation)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 2.8))
gq["answer"].value_counts().reindex(list("ABCD"), fill_value=0).plot.bar(ax=a1, color="#4c72b0", rot=0)
a1.set(title="Position of the correct option (after shuffling)", ylabel="questions")
gq["level"].value_counts().plot.bar(ax=a2, color="#dd8452", rot=0); a2.set(title="Bloom level requested by the model")
fig.tight_layout(); plt.show()

## 4. Verify (verifier)

From cheapest to most expensive:
1. **quote check** (code, free): every sentence of `source_quote` must be found in the section
   (fuzzy match ≥ 0.85) - otherwise `quote_not_found` and the verifier is not called;
2. **answer-back** (model 2): the verifier sees only the text around the quote and the options -
   **not** the answer - and must pick the correct one: a different choice gives `answer_mismatch`,
   no supported option (or several) gives `unanswerable`;
3. **warnings** (not rejected): `answer_leak` when the correct option is much longer than the others;
4. **dedupe**: near-identical questions are marked `duplicate`.

In [ ]:
before, start, t0 = snapshot(ver_llm), len(ver_llm.calls), time.perf_counter()
for q in qs:
    verify_one(q, by_title[q.section_title], ver_llm)
mark_duplicates(qs, use_st=False)
show_usage("4 verify", ver_llm, before, time.perf_counter() - t0, len(qs))
print("OUTPUT - prompt and raw answer of the first verifier call (questions rejected by the quote check are not sent):")
show_calls(ver_llm, start)

vq = pd.DataFrame([{"qid": q.qid, "quote found": quote_in_source(q.mcq.source_quote, by_title[q.section_title].text),
                    "generator answer": "ABCD"[q.mcq.answer_index],
                    "verifier answer": "-" if q.verifier_answer is None else ("none" if q.verifier_answer == -1 else "ABCD"[q.verifier_answer]),
                    "result": ";".join(q.rejected) or "accepted", "warnings": ";".join(q.warnings),
                    "verifier reason": q.verifier_reason[:90]} for q in qs])
display(vq)

if qs:   # what the verifier actually reads: the text around the quote, not the whole section
    q = qs[0]; ctx = find_context(q.mcq.source_quote, by_title[q.section_title].text, 300)
    quote = q.mcq.source_quote.strip()
    print("Context sent to the verifier for", q.qid, "(quote between >>> <<<):\n")
    print(ctx.replace(quote, f">>>{quote}<<<") if quote in ctx else ctx)

fig, ax = plt.subplots(figsize=(7, 2.6))
vq["result"].value_counts().plot.barh(ax=ax, color=["#55a868" if r == "accepted" else "#c44e52" for r in vq["result"].value_counts().index])
ax.set(title=f"Verification result ({sum(q.ok for q in qs)}/{len(qs)} accepted)", xlabel="questions")
fig.savefig(FIG_DIR / "verify.png", bbox_inches="tight"); plt.show()

## 5. Judge (LLM-as-judge, evaluation only)

The judge grades each **accepted** question on the same 0/1 rubric as the human rater (`eval/rubric.md`):
`factual`, `answerable`, `single_correct`, `distractors`, `level_match`. It is not part of the product:
it is checked against human ratings (Cohen's kappa) to know whether it can replace them.

In [ ]:
before, start, t0 = snapshot(judge_llm), len(judge_llm.calls), time.perf_counter()
scores = {}
for q in [q for q in qs if q.ok]:
    js = judge_one(judge_llm, q, by_title[q.section_title])
    if js:
        scores[q.qid] = {k: getattr(js, k) for k in RUBRIC_KEYS} | {"comment": js.comment[:80]}
show_usage("5 judge", judge_llm, before, time.perf_counter() - t0, len(scores))
print("OUTPUT - prompt and raw answer of the first judge call:")
show_calls(judge_llm, start)
jq = pd.DataFrame.from_dict(scores, orient="index")
display(jq)

if len(jq):
    m = jq[RUBRIC_KEYS].astype(int)
    fig, ax = plt.subplots(figsize=(7, 0.45 * len(m) + 1))
    ax.imshow(m.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(RUBRIC_KEYS)), RUBRIC_KEYS, rotation=20); ax.set_yticks(range(len(m)), m.index)
    for (r, c), v in pd.DataFrame(m.values).stack().items():
        ax.text(c, r, v, ha="center", va="center")
    ax.set_title("Judge score per question (green = 1)"); fig.savefig(FIG_DIR / "judge.png", bbox_inches="tight"); plt.show()

## 6. Consumption of every step

In [ ]:
cost = pd.DataFrame(LEDGER.values()).set_index("step")
display(cost)
total = cost["cost $"].sum()
print(f"total: ${total:.5f}, {cost['tokens in'].sum() + cost['tokens out'].sum():,} tokens, "
      f"{cost['seconds'].sum():.1f} s for {len(qs)} questions"
      + (f" -> ${100 * total / len(qs):.4f} per 100 questions" if qs else ""))

fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(14, 3))
cost[["tokens in", "tokens out"]].plot.bar(stacked=True, ax=a1, rot=20, color=["#8fa8d0", "#4c72b0"]); a1.set(title="Tokens per step")
cost["cost $"].plot.bar(ax=a2, rot=20, color="#c44e52"); a2.set(title="Cost per step ($)")
cost["seconds"].plot.bar(ax=a3, rot=20, color="#dd8452"); a3.set(title="Time per step (s)")
fig.tight_layout(); fig.savefig(FIG_DIR / "cost_demo.png", bbox_inches="tight"); plt.show()

## 7. Explore a finished evaluation run (no model call)

Reads what `python -m eval.run_eval ... --out reports/eval` wrote: `questions.csv` (every question with
its verification result and judge scores), `runs/*/*/*/cell_stats.json` (tokens, cost and time per
cell) and, once you have rated it, `labeling_sheet.csv` (judge vs human).

In [ ]:
from eval.kappa import kappa_table, read_sheet

run = Path(RUN_DIR)
if not (run / "questions.csv").exists():
    print(f"{run}/questions.csv not found - run eval.run_eval first.")
else:
    q = read_sheet(run / "questions.csv")
    q["result"] = q["rejected"].fillna("").replace("", "accepted").str.split(";").str[0]
    print(f"{len(q)} questions, {(q['result'] == 'accepted').mean():.1%} accepted")
    display(pd.crosstab([q["config"], q["strategy"]], q["result"]))

    judge_cols = [f"judge_{k}" for k in RUBRIC_KEYS if f"judge_{k}" in q]
    if judge_cols:
        means = q.groupby("strategy")[judge_cols].mean().rename(columns=lambda c: c.removeprefix("judge_"))
        ax = means.T.plot.bar(figsize=(10, 3.2), rot=0, ylim=(0, 1.05), title="Judge: share of questions passing each criterion")
        ax.legend(title="strategy", fontsize=8); ax.figure.savefig(FIG_DIR / "judge_by_strategy.png", bbox_inches="tight"); plt.show()

    import json
    st = pd.DataFrame([{"strategy": f.parts[-3], **json.loads(f.read_text())} for f in run.glob("runs/*/*/*/cell_stats.json")])
    if len(st):
        g = st.groupby("strategy")[["questions", "gen_cost_usd", "verify_cost_usd", "judge_cost_usd",
                                    "gen_seconds", "verify_seconds", "judge_seconds"]].sum()
        per100 = g[["gen_cost_usd", "verify_cost_usd", "judge_cost_usd"]].div(g["questions"], axis=0) * 100
        minutes = g[["gen_seconds", "verify_seconds", "judge_seconds"]] / 60
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3))
        per100.rename(columns=lambda c: c.split("_")[0]).plot.bar(stacked=True, ax=a1, rot=0, title="Cost per 100 questions ($)")
        minutes.rename(columns=lambda c: c.split("_")[0]).plot.bar(stacked=True, ax=a2, rot=0, title="Time (minutes)")
        fig.tight_layout(); fig.savefig(FIG_DIR / "cost_by_strategy.png", bbox_inches="tight"); plt.show()

    sheet = run / "labeling_sheet.csv"
    if sheet.exists() and (run / "judge_labels.csv").exists():
        human = read_sheet(sheet).dropna(subset=RUBRIC_KEYS)
        if len(human):
            kt = kappa_table(human[["id"] + RUBRIC_KEYS], read_sheet(run / "judge_labels.csv"))
            display(kt)
            ax = kt.set_index("criterion")[["agreement", "kappa"]].plot.bar(figsize=(9, 3), rot=0, ylim=(-0.1, 1.05),
                                                                             title=f"Judge vs human ({len(human)} rated questions)")
            ax.axhline(0.6, ls="--", c="grey"); ax.figure.savefig(FIG_DIR / "judge_vs_human.png", bbox_inches="tight"); plt.show()
        else:
            print("No human rating yet in labeling_sheet.csv.")